# Module 16 — Human–Multi-Agent Organizations

> **SDKs:** `pydantic`, `dataclasses`

| Part | Topic |
|------|-------|
| **1** | The Manager Agent — delegation and accountability |
| **2** | Delegation & Accountability — typed contracts |
| **3** | Human-in-the-Loop Patterns — approval workflows |


---
## Part 1 — The Manager Agent

A Manager Agent decomposes high-level goals, delegates sub-tasks to specialists, aggregates results, and determines whether to escalate to humans. It is NOT an all-knowing oracle — it is a coordinator.

In [1]:
from dataclasses import dataclass, field
from typing import Optional, Literal
from pydantic import BaseModel

class DelegationContract(BaseModel):
    """The formal contract between Manager and Specialist agents."""
    task_id: str
    task_description: str
    specialist: str
    allowed_tools: list[str]
    max_steps: int
    required_output_schema: str   # name of Pydantic model
    deadline_seconds: int

class SpecialistResult(BaseModel):
    task_id: str
    specialist: str
    output: dict
    tool_calls_made: int
    duration_s: float
    success: bool
    confidence: Literal["LOW","MEDIUM","HIGH"]

class ManagerAgent:
    """
    Decomposes goals → delegates to specialists → aggregates → decides.
    Does NOT execute tools itself (only reads specialist outputs).
    """
    def __init__(self, name: str, specialists: list[str]):
        self.name = name
        self.specialists = specialists

    def decompose(self, goal: str) -> list[DelegationContract]:
        """Break goal into specialist sub-tasks."""
        print(f"  [Manager] Decomposing goal: '{goal[:60]}'")
        return [
            DelegationContract(task_id="t-001", task_description="Gather telemetry evidence",
                specialist="ObservabilitySpecialist", allowed_tools=["query_metrics","search_logs"],
                max_steps=5, required_output_schema="TelemetryArtifact", deadline_seconds=30),
            DelegationContract(task_id="t-002", task_description="Gather deployment evidence",
                specialist="DeploymentSpecialist", allowed_tools=["get_deployment","search_commits"],
                max_steps=3, required_output_schema="DeploymentArtifact", deadline_seconds=20),
        ]

    def aggregate(self, results: list[SpecialistResult]) -> dict:
        """Synthesise specialist outputs."""
        all_ok = all(r.success for r in results)
        avg_confidence = "HIGH" if all(r.confidence == "HIGH" for r in results) else "MEDIUM"
        return {
            "all_tasks_completed": all_ok,
            "combined_confidence": avg_confidence,
            "requires_escalation": not all_ok,
            "summary": f"All {len(results)} specialists completed. Confidence: {avg_confidence}.",
        }

manager = ManagerAgent("IncidentManager", ["ObservabilitySpecialist", "DeploymentSpecialist"])

print("🏛️  Manager Agent Demo")
print("=" * 60)
contracts = manager.decompose("Investigate EU checkout conversion drop of 38%")
print(f"\n  Decomposed into {len(contracts)} sub-tasks:")
for c in contracts:
    print(f"    [{c.task_id}] → {c.specialist}: '{c.task_description}'")
    print(f"           tools={c.allowed_tools}  max_steps={c.max_steps}")

# Simulate specialist results
results = [
    SpecialistResult(task_id="t-001", specialist="ObservabilitySpecialist",
        output={"error_rate": 0.31, "evidence_id": "EV-001"}, tool_calls_made=2, duration_s=3.2, success=True, confidence="HIGH"),
    SpecialistResult(task_id="t-002", specialist="DeploymentSpecialist",
        output={"version": "v2.1", "evidence_id": "EV-003"}, tool_calls_made=1, duration_s=1.8, success=True, confidence="HIGH"),
]

aggregate = manager.aggregate(results)
print(f"\n  Aggregated result:")
for k, v in aggregate.items():
    print(f"    {k}: {v}")


🏛️  Manager Agent Demo
  [Manager] Decomposing goal: 'Investigate EU checkout conversion drop of 38%'

  Decomposed into 2 sub-tasks:
    [t-001] → ObservabilitySpecialist: 'Gather telemetry evidence'
           tools=['query_metrics', 'search_logs']  max_steps=5
    [t-002] → DeploymentSpecialist: 'Gather deployment evidence'
           tools=['get_deployment', 'search_commits']  max_steps=3

  Aggregated result:
    all_tasks_completed: True
    combined_confidence: HIGH
    requires_escalation: False
    summary: All 2 specialists completed. Confidence: HIGH.


---
## Part 2 & 3 — Delegation Contracts & Human Approval Workflows

Typed DelegationContracts create accountability chains. Human approval workflows define how agents escalate when autonomy boundaries are reached.

In [2]:
from datetime import datetime, timezone, timedelta
from pydantic import BaseModel
from typing import Literal, Optional
import hashlib, time

class ApprovalRequest(BaseModel):
    request_id: str
    requester: str       # agent that created this
    action: str
    justification: str
    risk_level: Literal["LOW","MEDIUM","HIGH","CRITICAL"]
    expires_at: str
    idempotency_key: str

class ApprovalResponse(BaseModel):
    request_id: str
    approved: bool
    approver: str
    reason: str
    timestamp: str

def create_approval_request(agent: str, action: str, justification: str, risk: str) -> ApprovalRequest:
    req_id = "apr-" + hashlib.sha256(f"{agent}:{action}".encode()).hexdigest()[:8]
    expires = datetime.now(timezone.utc) + timedelta(minutes=30)
    return ApprovalRequest(
        request_id=req_id,
        requester=agent,
        action=action,
        justification=justification,
        risk_level=risk,
        expires_at=expires.isoformat(),
        idempotency_key=hashlib.sha256(f"{req_id}:{action}".encode()).hexdigest()[:16],
    )

# ─── HITL workflow ────────────────────────────────────────────────────────────
print("🤝  Human–Agent Approval Workflow Demo")
print("=" * 60)

# Tiered approval based on risk level
APPROVAL_MATRIX = {
    "LOW":      "auto-approve (no human needed)",
    "MEDIUM":   "Team Lead approval",
    "HIGH":     "Senior Engineer approval + second opinion",
    "CRITICAL": "VP Engineering approval required",
}

test_requests = [
    ("ObsAgent",    "Read-only metrics query",   "Need evidence for investigation", "LOW"),
    ("AnalystAgent","Propose revert v2.1→v2.0",  "3DS broken by v2.1 deployment",  "HIGH"),
    ("ExecAgent",   "Delete production database", "Cleanup stale tables",           "CRITICAL"),
]

for agent, action, justification, risk in test_requests:
    req = create_approval_request(agent, action, justification, risk)
    required_approver = APPROVAL_MATRIX[risk]
    
    print(f"\n  [{risk}] {agent} wants to: '{action}'")
    print(f"    Request ID       : {req.request_id}")
    print(f"    Idempotency Key  : {req.idempotency_key}")
    print(f"    Expires At       : {req.expires_at[11:16]} UTC")
    print(f"    Requires Approval: {required_approver}")
    
    if risk in ("LOW",):
        print(f"    ✅  Auto-approved — low risk action")
    elif risk == "CRITICAL":
        print(f"    🛑  Escalated to VP Engineering — no auto-execution possible")
    else:
        # Simulate human approval
        response = ApprovalResponse(
            request_id=req.request_id,
            approved=True,
            approver="sarah.chen@northstar.com",
            reason="Evidence is solid. Revert is safe.",
            timestamp=datetime.now(timezone.utc).isoformat(),
        )
        print(f"    ✅  Approved by {response.approver}: '{response.reason}'")


🤝  Human–Agent Approval Workflow Demo

  [LOW] ObsAgent wants to: 'Read-only metrics query'
    Request ID       : apr-c9a1c35b
    Idempotency Key  : 1bf115ab99445afb
    Expires At       : 05:42 UTC
    Requires Approval: auto-approve (no human needed)
    ✅  Auto-approved — low risk action

  [HIGH] AnalystAgent wants to: 'Propose revert v2.1→v2.0'
    Request ID       : apr-08c10699
    Idempotency Key  : 9d133f0d452b90cf
    Expires At       : 05:42 UTC
    Requires Approval: Senior Engineer approval + second opinion
    ✅  Approved by sarah.chen@northstar.com: 'Evidence is solid. Revert is safe.'

  [CRITICAL] ExecAgent wants to: 'Delete production database'
    Request ID       : apr-e9e09036
    Idempotency Key  : 4d3d9fb6ccf032de
    Expires At       : 05:42 UTC
    Requires Approval: VP Engineering approval required
    🛑  Escalated to VP Engineering — no auto-execution possible
